# Add a controller-tuning problem

Build a small deterministic mass-spring-damper problem in four steps: simulate,
configure, evaluate, and inspect the result. Direct construction needs no registry.
The final link explains registration when packaging a family for discovery.

Install TuneControl before running these cells in order. Only the base package
is needed; there is no optimization dependency in this example.


## 1. Imports
We use a dataclass for configuration and subclass the common task interface.


In [ ]:
from dataclasses import dataclass
import torch

from tunecontrol import Task
from tunecontrol.tasks.base import validate_bounds
from tunecontrol.tasks.module_utils import detach_trajectory


## 2. Simulate a controlled system

A force-actuated mass starts at rest at position zero and tracks a 1 m reference.
The two parameters are proportional and derivative gains. The update computes
velocity first, then position using the new velocity (semi-implicit Euler).

Record 200 state/input pairs at 0.02 s intervals, from 0 to 3.98 s. Each row
contains the state and the control computed from it, before the update. The
trajectory uses named columns and explicit SI units.


In [ ]:
@dataclass
class MassSpringSimulator:
    dt: float = 0.02
    horizon: int = 200
    mass: float = 1.0
    damping: float = 0.2
    stiffness: float = 1.0
    reference: float = 1.0

    def simulate(self, theta: torch.Tensor):
        Kp, Kd = float(theta[0]), float(theta[1])
        pos = torch.tensor(0.0, dtype=torch.float64)
        vel = torch.tensor(0.0, dtype=torch.float64)

        time = []
        positions = []
        velocities = []
        controls = []
        references = []

        for step in range(self.horizon):
            t = step * self.dt
            error = self.reference - pos
            control = Kp * error - Kd * vel
            accel = (control - self.damping * vel - self.stiffness * pos) / self.mass
            time.append(t)
            positions.append(pos.item())
            velocities.append(vel.item())
            controls.append(control.item())
            references.append(self.reference)

            vel = vel + accel * self.dt
            pos = pos + vel * self.dt

        trajectory = {
            "time": torch.tensor(time, dtype=torch.float64),
            "states": torch.tensor(list(zip(positions, velocities)), dtype=torch.float64),
            "inputs": torch.tensor(controls, dtype=torch.float64).reshape(-1, 1),
            "state_names": ("position", "velocity"),
            "state_units": ("m", "m/s"),
            "input_names": ("force",),
            "input_units": ("N",),
            "reference": torch.tensor(references, dtype=torch.float64),
        }
        return trajectory


## 3. Task wrapper
We wrap the simulator in a TuneControl `Task`, define bounds, and compute
a simple integral of squared error (with a small control penalty) as the objective.


In [ ]:
@dataclass(frozen=True)
class MassSpringConfig:
    mass: float = 1.0
    damping: float = 0.2
    stiffness: float = 1.0

    def __post_init__(self):
        import math
        for name in ("mass", "damping", "stiffness"):
            value = getattr(self, name)
            if not math.isfinite(value) or value <= 0:
                raise ValueError(f"{name} must be finite and positive")


class MassSpring(Task):
    config_type = MassSpringConfig
    name = "MassSpring"

    @classmethod
    def available_configs(cls):
        return [MassSpringConfig()]

    def __init__(self, config: MassSpringConfig | None = None) -> None:
        config = MassSpringConfig() if config is None else config
        if not isinstance(config, MassSpringConfig):
            raise TypeError("config must be MassSpringConfig")
        self.config = config
        self.sim = MassSpringSimulator(mass=config.mass, damping=config.damping, stiffness=config.stiffness)
        self.dim = 2
        self.bounds = torch.tensor([[0.0, 0.0], [20.0, 5.0]], dtype=torch.float64)
        validate_bounds(self.bounds, self.dim)
        self.is_minimization = True

    def _evaluate(self, theta: torch.Tensor):
        trajectory = self.sim.simulate(theta)
        position = trajectory["states"][:, 0]
        control = trajectory["inputs"][:, 0]
        ref = trajectory["reference"]
        error = ref - position
        dt = self.sim.dt
        ise = torch.sum(error.pow(2)) * dt
        r_control = 0.01 * torch.sum(control.pow(2)) * dt
        value = ise + r_control
        info = {
            "theta": theta.detach().clone(),
            "trajectory": detach_trajectory(trajectory, dtype=theta.dtype, device=theta.device),
        }
        return value.to(dtype=theta.dtype), info


## 4. Construct and evaluate
Create a configuration, construct the problem, and evaluate a controller at the midpoint of its bounds.


In [ ]:
config = MassSpringConfig(mass=1.0, damping=0.2, stiffness=1.0)
problem = MassSpring(config)
theta = problem.bounds.mean(dim=0)
value, info = problem.evaluate(theta)

print("Configuration:", problem.config)
print("Controller:", theta.tolist())
print("Cost:", value.item())
print("Trajectory keys:", sorted(info["trajectory"]))


## 5. Add the family to a package

Once the problem is ready to share, follow the [contribution guide](../docs/task_module_architecture.md#add-a-family) to expose it through package registration.
